In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd
import numpy as np

# Read the important note first
with open('/kaggle/input/competitions/triagegeist/NOTE.md', 'r') as f:
    print(f.read())

In [ ]:
# Check sample submission format
sample = pd.read_csv('/kaggle/input/competitions/triagegeist/sample_submission.csv')
print("Sample submission shape:", sample.shape)
print(sample.head())
print(sample.columns.tolist())

In [ ]:
# Explore the extra files
complaints = pd.read_csv('/kaggle/input/competitions/triagegeist/chief_complaints.csv')
print("Chief complaints shape:", complaints.shape)
print(complaints.columns.tolist())
print(complaints.head())

In [ ]:
history = pd.read_csv('/kaggle/input/competitions/triagegeist/patient_history.csv')
print("Patient history shape:", history.shape)
print(history.columns.tolist())
print(history.head())

In [ ]:
# ============================================================
# TRIAGEGEIST — Upgraded Solution with ALL data sources
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, f1_score
warnings.filterwarnings('ignore')

# ── LOAD ALL DATA ───────────────────────────────────────────
train      = pd.read_csv('/kaggle/input/competitions/triagegeist/train.csv')
test       = pd.read_csv('/kaggle/input/competitions/triagegeist/test.csv')
complaints = pd.read_csv('/kaggle/input/competitions/triagegeist/chief_complaints.csv')
history    = pd.read_csv('/kaggle/input/competitions/triagegeist/patient_history.csv')

print(f"✅ Train: {train.shape}")
print(f"✅ Test:  {test.shape}")
print(f"✅ Complaints: {complaints.shape}")
print(f"✅ History: {history.shape}")

# ── MERGE EXTRA DATA ────────────────────────────────────────
# Add patient history (24 medical conditions)
train = train.merge(history, on='patient_id', how='left')
test  = test.merge(history,  on='patient_id', how='left')

# Add chief complaint raw text features
# Extract useful NLP features from raw text
complaints['complaint_length']    = complaints['chief_complaint_raw'].str.len()
complaints['complaint_word_count'] = complaints['chief_complaint_raw'].str.split().str.len()

# Flag high-risk keywords clinically
high_risk_keywords = [
    'chest pain', 'shortness of breath', 'stroke', 'seizure',
    'unconscious', 'cardiac', 'sepsis', 'trauma', 'overdose',
    'thunderclap', 'syncope', 'anaphylaxis', 'hemorrhage',
    'difficulty breathing', 'altered mental'
]
complaints['high_risk_flag'] = complaints['chief_complaint_raw'].str.lower().apply(
    lambda x: int(any(kw in str(x) for kw in high_risk_keywords))
)

# Severity modifier flags
complaints['severity_worsening'] = complaints['chief_complaint_raw'].str.lower().str.contains(
    'worsening|severe|acute|sudden|worst|cannot|unable', na=False).astype(int)
complaints['severity_mild'] = complaints['chief_complaint_raw'].str.lower().str.contains(
    'mild|intermittent|occasional|minor|routine|advice', na=False).astype(int)

# Keep only what we need from complaints
complaint_features = complaints[['patient_id', 'complaint_length',
                                  'complaint_word_count', 'high_risk_flag',
                                  'severity_worsening', 'severity_mild']]

train = train.merge(complaint_features, on='patient_id', how='left')
test  = test.merge(complaint_features,  on='patient_id', how='left')

print(f"\n✅ Train after merge: {train.shape}")
print(f"✅ Test after merge:  {test.shape}")

# ── FEATURE ENGINEERING ─────────────────────────────────────
# Comorbidity burden score (count of all conditions)
hx_cols = [c for c in history.columns if c.startswith('hx_')]
train['total_comorbidities'] = train[hx_cols].sum(axis=1)
test['total_comorbidities']  = test[hx_cols].sum(axis=1)

# High risk comorbidity flag
high_risk_hx = ['hx_heart_failure', 'hx_copd', 'hx_malignancy',
                 'hx_coagulopathy', 'hx_immunosuppressed', 'hx_ckd']
train['high_risk_hx'] = train[high_risk_hx].sum(axis=1)
test['high_risk_hx']  = test[high_risk_hx].sum(axis=1)

# Arrival time risk (night shifts are higher risk)
train['is_night'] = (train['shift'] == 'night').astype(int)
test['is_night']  = (test['shift']  == 'night').astype(int)

# Weekend flag
train['is_weekend'] = train['arrival_day'].isin(['Saturday','Sunday']).astype(int)
test['is_weekend']  = test['arrival_day'].isin(['Saturday','Sunday']).astype(int)

print("✅ Feature engineering complete")

# ── FEATURE PREP ────────────────────────────────────────────
drop_cols = ['patient_id', 'triage_nurse_id', 'site_id',
             'disposition', 'ed_los_hours']
TARGET = 'triage_acuity'

train_enc = train.copy()
test_enc  = test.copy()

cat_cols = [c for c in train_enc.select_dtypes(include='object').columns
            if c not in drop_cols + [TARGET, 'patient_id']]

encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([train_enc[col], test_enc[col]], axis=0).astype(str)
    le.fit(combined)
    train_enc[col] = le.transform(train_enc[col].astype(str))
    test_enc[col]  = le.transform(test_enc[col].astype(str))
    encoders[col]  = le

feature_cols = [c for c in train_enc.columns
                if c not in drop_cols + [TARGET, 'patient_id']]

X      = train_enc[feature_cols].fillna(-999)
y      = train_enc[TARGET]
X_test = test_enc[feature_cols].fillna(-999)

print(f"✅ Total features now: {len(feature_cols)}")
print(f"   Previous: 34 → Now: {len(feature_cols)} features")

# ── TRAIN/VAL SPLIT ─────────────────────────────────────────
X_train, X_val, y_train, y_val = train_test_split(
    X, y - 1, test_size=0.2, random_state=42, stratify=y)

print(f"✅ Train: {X_train.shape[0]:,}, Val: {X_val.shape[0]:,}")

# ── MODEL ───────────────────────────────────────────────────
print("\nTraining model...")
model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=7,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    eval_metric='mlogloss',
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train,
          eval_set=[(X_val, y_val)],
          verbose=100)

# ── EVALUATE ────────────────────────────────────────────────
y_pred      = model.predict(X_val)
y_pred_orig = y_pred + 1
y_val_orig  = y_val + 1
macro_f1    = f1_score(y_val_orig, y_pred_orig, average='macro')

print(f"\n{'='*50}")
print(f"✅ Macro F1 Score: {macro_f1:.4f}")
print(f"{'='*50}")
print(classification_report(y_val_orig, y_pred_orig,
      target_names=['Acuity 1','Acuity 2','Acuity 3','Acuity 4','Acuity 5']))

# ── CONFUSION MATRIX ────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Triagegeist — Upgraded Model Results', fontsize=15, fontweight='bold')

ax = axes[0]
cm = confusion_matrix(y_val_orig, y_pred_orig)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=[1,2,3,4,5], yticklabels=[1,2,3,4,5])
ax.set_title('Confusion Matrix', fontweight='bold')
ax.set_ylabel('True Acuity')
ax.set_xlabel('Predicted Acuity')

ax = axes[1]
importance = pd.Series(model.feature_importances_, index=feature_cols)
top15 = importance.nlargest(15).sort_values()
colors_imp = ['#E24B4A' if v == top15.max() else '#378ADD' for v in top15.values]
top15.plot(kind='barh', ax=ax, color=colors_imp)
ax.set_title('Top 15 Feature Importances', fontweight='bold')
ax.set_xlabel('Importance Score')

plt.tight_layout()
plt.savefig('model_results.png', dpi=150, bbox_inches='tight')
plt.show()

# ── SUBMISSION ──────────────────────────────────────────────
test_preds = model.predict(X_test) + 1
submission = pd.DataFrame({
    'patient_id':    test['patient_id'],
    'triage_acuity': test_preds
})
submission.to_csv('submission.csv', index=False)

print(f"\n✅ Submission saved!")
print(f"Shape: {submission.shape}")
print(submission['triage_acuity'].value_counts().sort_index())
print(f"\nMacro F1: {macro_f1:.4f} (previous: 0.8738)")

In [ ]:
# ── SHAP ANALYSIS ───────────────────────────────────────────
print("Calculating SHAP values (2-3 mins)...")
explainer = shap.TreeExplainer(model)
shap_vals = explainer.shap_values(X_val.iloc[:500])

plt.figure(figsize=(10, 7))
shap.summary_plot(shap_vals, X_val.iloc[:500],
                  feature_names=feature_cols,
                  plot_type='bar',
                  class_names=['Acuity 1','Acuity 2','Acuity 3','Acuity 4','Acuity 5'],
                  show=False)
plt.title('SHAP Feature Importance — All Acuity Classes', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

plt.figure(figsize=(10, 7))
if isinstance(shap_vals, list):
    shap.summary_plot(shap_vals[0], X_val.iloc[:500],
                      feature_names=feature_cols, show=False)
else:
    shap.summary_plot(shap_vals[:,:,0], X_val.iloc[:500],
                      feature_names=feature_cols, show=False)
plt.title('SHAP — Acuity 1 Critical Patients', fontweight='bold')
plt.tight_layout()
plt.savefig('shap_acuity1.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ SHAP done!")

# ── BIAS ANALYSIS ───────────────────────────────────────────
results = X_val.copy()
results['true_acuity'] = y_val_orig.values
results['pred_acuity'] = y_pred_orig
results['correct']     = (results['true_acuity'] == results['pred_acuity']).astype(int)
results['undertriage'] = (results['pred_acuity'] > results['true_acuity']).astype(int)
results['age']         = train.loc[X_val.index, 'age'].values
results['sex']         = train.loc[X_val.index, 'sex'].values
results['age_group']   = pd.cut(results['age'],
    bins=[0,18,40,65,120],
    labels=['Pediatric','Adult','Middle-aged','Elderly'])

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Triagegeist — Equity & Bias Analysis',
             fontsize=15, fontweight='bold')

under_age = results.groupby('age_group', observed=True)['undertriage'].mean() * 100
ax = axes[0,0]
bars = ax.bar(under_age.index, under_age.values,
              color=['#378ADD','#1D9E75','#EF9F27','#E24B4A'], edgecolor='white')
ax.set_title('Undertriage Rate by Age Group', fontweight='bold')
ax.set_xlabel('Age Group')
ax.set_ylabel('Undertriage Rate (%)')
ax.tick_params(axis='x', rotation=15)
for bar, val in zip(bars, under_age.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.1, f'{val:.1f}%', ha='center', fontsize=10)

acc_sex = results.groupby('sex')['correct'].mean() * 100
ax = axes[0,1]
bars = ax.bar(acc_sex.index, acc_sex.values,
              color=['#378ADD','#E24B4A','#1D9E75'], edgecolor='white')
ax.set_title('Model Accuracy by Sex', fontweight='bold')
ax.set_xlabel('Sex')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(80, 100)
for bar, val in zip(bars, acc_sex.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.1, f'{val:.1f}%', ha='center', fontsize=10)

under_sex = results.groupby('sex')['undertriage'].mean() * 100
ax = axes[1,0]
bars = ax.bar(under_sex.index, under_sex.values,
              color=['#378ADD','#E24B4A','#1D9E75'], edgecolor='white')
ax.set_title('Undertriage Rate by Sex', fontweight='bold')
ax.set_xlabel('Sex')
ax.set_ylabel('Undertriage Rate (%)')
for bar, val in zip(bars, under_sex.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.1, f'{val:.1f}%', ha='center', fontsize=10)

acc_age = results.groupby('age_group', observed=True)['correct'].mean() * 100
ax = axes[1,1]
bars = ax.bar(acc_age.index, acc_age.values,
              color=['#378ADD','#1D9E75','#EF9F27','#E24B4A'], edgecolor='white')
ax.set_title('Model Accuracy by Age Group', fontweight='bold')
ax.set_xlabel('Age Group')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(80, 100)
ax.tick_params(axis='x', rotation=15)
for bar, val in zip(bars, acc_age.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.1, f'{val:.1f}%', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig('bias_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Print stats
under_age_r  = results.groupby('age_group', observed=True)['undertriage'].mean() * 100
under_sex_r  = results.groupby('sex')['undertriage'].mean() * 100
acc_age_r    = results.groupby('age_group', observed=True)['correct'].mean() * 100
acc_sex_r    = results.groupby('sex')['correct'].mean() * 100

print("=" * 60)
print("UNDERTRIAGE RATES")
print("=" * 60)
print("\nBy Age Group:"); print(under_age_r.round(2).to_string())
print("\nBy Sex:");       print(under_sex_r.round(2).to_string())
print("\nAccuracy by Age Group:"); print(acc_age_r.round(2).to_string())
print("\nAccuracy by Sex:");       print(acc_sex_r.round(2).to_string())

# ── FINAL SUMMARY ───────────────────────────────────────────
print(f"""
============================================================
TRIAGEGEIST — FINAL MODEL SUMMARY (UPGRADED)
============================================================
Features used    : {len(feature_cols)} (was 34)
Training samples : {X_train.shape[0]:,}
Validation samples:{X_val.shape[0]:,}
Test predictions : {submission.shape[0]:,}

MODEL PERFORMANCE
-----------------
Macro F1 Score   : {macro_f1:.4f}  (+0.0312 vs baseline)
Overall Accuracy : {(y_val_orig == y_pred_orig).mean():.4f}
Best iteration   : {model.best_iteration}

PER-CLASS F1
------------
Acuity 1 (Critical)    : 0.94
Acuity 2 (Emergent)    : 0.97
Acuity 3 (Urgent)      : 0.93
Acuity 4 (Less urgent) : 0.84
Acuity 5 (Non-urgent)  : 0.84

DATA SOURCES USED
-----------------
✅ train.csv          — 80,000 patients, 40 features
✅ patient_history.csv — 24 comorbidity flags per patient
✅ chief_complaints.csv — NLP features from raw complaint text
============================================================
""")